# mEC open-field spatial tuning (stage 1, classic scores)

Reads the merged tables and shards of one run of `run_spatial.py --engine of --dataset mec`; it never
opens a data_dic (`RUN_MODE = 'load'`). Write-up: `SPATIAL_TUNING.md`. Galleries of every unit are in the
run's `figures/` folder (`spatial_plots.of_gallery_pdf`).

In [ ]:
import os, sys, glob, pickle
import numpy as np, pandas as pd
sys.path.insert(0, os.path.abspath('.'))
import spatial_plots as sp
RUN_MODE = 'load'
RUN = '/ceph/behrens/adam_harris/Taskspace_abstraction_mEC/data/spatial/of/8a49b0e2d93c'
units = pd.read_csv(sorted(glob.glob(f'{RUN}/merged/of_units*.csv'))[-1])
sessions = pd.read_csv(sorted(glob.glob(f'{RUN}/merged/of_sessions*.csv'))[-1])
rotations = pd.read_csv(sorted(glob.glob(f'{RUN}/merged/of_rotations*.csv'))[-1])
sp.setup_style()

## Class counts per session (broad eligible units) against the one-shift-for-all chance distribution

In [ ]:
b = sessions[(sessions.pool == 'broad') & sessions.cls.isin(['grid', 'border', 'spatial', 'hd', 'speed'])]
table = b.pivot_table(index=['recday', 'session', 'session_type', 'n_units'], columns='cls', values=['observed', 'chance_p95', 'p'], aggfunc='first')
table

In [ ]:
fig = sp.plot_class_counts(sessions)

## Border cells, and HD conjunction

In [ ]:
u = units[units.eligible & ~units.narrow]
u[u.is_border][['recday', 'session', 'unit', 'border', 'border_wall', 'is_hd', 'pxd_r', 'rate_hz']]

## Grid scores: the distribution against each session's pooled shuffle p95

In [ ]:
rows = []
for f in sorted(glob.glob(f'{RUN}/shards/*.pkl')):
    sh = pickle.load(open(f, 'rb'))
    for s, ss in sh['sessions'].items():
        pb = np.array(ss['pools']['broad'], int)
        if len(pb):
            rows.append(dict(recday=sh['recday'], session=s, n=len(pb), p95=np.nanpercentile(ss['null']['grid'][:, pb], 95),
                             max_grid=np.nanmax(ss['real']['grid'][pb]), above_p95=int(np.nansum(ss['real']['grid'][pb] > np.nanpercentile(ss['null']['grid'][:, pb], 95)))))
pd.DataFrame(rows)

## Rotation between the two arena sessions of one sort (bp01 04-02 to 04-03)

In [ ]:
rotations.T

## Example gallery page (every unit is in the PDFs)

In [ ]:
import matplotlib.pyplot as plt
sh = pickle.load(open(f'{RUN}/shards/bp01_20240402_20240403.pkl', 'rb'))
ss = sh['sessions'][4]
order = np.argsort(-np.nan_to_num(ss['real']['border'], nan=-9))[:3]
fig = plt.figure(figsize=(7.5, 5.5))
fig.subplots_adjust(hspace=0.75, wspace=0.35)
for row, u_ in enumerate(order):
    axs = [fig.add_subplot(3, 4, row * 4 + 1), fig.add_subplot(3, 4, row * 4 + 2),
           fig.add_subplot(3, 4, row * 4 + 3, projection='polar'), fig.add_subplot(3, 4, row * 4 + 4)]
    sp.plot_of_unit(axs, ss, u_)